In [ ]:
import json
import csv
import logging
from collections import Counter
from typing import List, Dict, Optional
import tkinter as tk
from tkinter import ttk, messagebox

# LOGGING SETUP
logging.basicConfig(
    filename="refreshfood.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# DATA MANAGER
class ReFreshFoodManager:
    def __init__(self, file_dati: str = "dati_refresh.json") -> None:
        self.file_dati: str = file_dati
        self.items: List[Dict[str, any]] = []
        self.carica_dati()

    def salva_dati(self) -> None:
        with open(self.file_dati, "w", encoding="utf-8") as f:
            json.dump(self.items, f, indent=4, ensure_ascii=False)
        logging.info("Dati salvati nel file JSON.")

    def carica_dati(self) -> None:
        try:
            with open(self.file_dati, "r", encoding="utf-8") as f:
                self.items = json.load(f)
            logging.info("Dati caricati correttamente.")
        except FileNotFoundError:
            self.items = []
            logging.warning("File JSON non trovato, creato nuovo archivio.")

    def aggiungi_elemento(self, nome: str, ingredienti: List[str], categoria: str, tempo_preparazione: int) -> None:
        elemento = {
            "nome": nome,
            "ingredienti": ingredienti,
            "categoria": categoria,
            "tempo_preparazione": tempo_preparazione
        }
        self.items.append(elemento)
        self.salva_dati()
        logging.info(f"Aggiunto elemento: {nome}")

    def elimina_elemento(self, indice: int) -> None:
        if 0 <= indice < len(self.items):
            nome = self.items[indice]['nome']
            self.items.pop(indice)
            self.salva_dati()
            logging.info(f"Eliminato elemento: {nome}")

    def ricerca(self, testo: str) -> List[Dict[str, any]]:
        testo = testo.lower()
        risultati = [
            elem for elem in self.items
            if testo in elem["nome"].lower()
            or testo in elem["categoria"].lower()
            or any(testo in ingr.lower() for ingr in elem["ingredienti"])
        ]
        logging.info(f"Ricerca effettuata: '{testo}' ({len(risultati)} risultati)")
        return risultati

    def filtro_avanzato(self, max_tempo: Optional[int], categoria: Optional[str], ingrediente: Optional[str]) -> List[Dict[str, any]]:
        risultati = self.items
        if max_tempo is not None:
            risultati = [e for e in risultati if e["tempo_preparazione"] <= max_tempo]
        if categoria:
            risultati = [e for e in risultati if categoria.lower() in e["categoria"].lower()]
        if ingrediente:
            risultati = [e for e in risultati if any(ingrediente.lower() in i.lower() for i in e["ingredienti"])]
        logging.info(f"Filtro avanzato applicato. Risultati: {len(risultati)}")
        return risultati

    def statistiche(self) -> Dict[str, any]:
        if not self.items:
            return {}
        totale = len(self.items)
        categorie = Counter([e["categoria"] for e in self.items])
        tempi = [e["tempo_preparazione"] for e in self.items]
        return {
            "totale": totale,
            "media_tempo": sum(tempi) / totale,
            "min_tempo": min(tempi),
            "max_tempo": max(tempi),
            "categorie": categorie
        }

    def ordina(self, chiave: str) -> None:
        self.items.sort(key=lambda e: e[chiave].lower() if isinstance(e[chiave], str) else e[chiave])
        self.salva_dati()
        logging.info(f"Ordinamento per '{chiave}' completato.")

    def suggerisci_simili(self, index: int, soglia: int = 2) -> List[Dict[str, any]]:
        """Trova ricette con almeno 'soglia' ingredienti in comune."""
        if not (0 <= index < len(self.items)):
            return []
        base = self.items[index]
        base_ingredienti = set(i.lower() for i in base["ingredienti"])
        suggerimenti = []
        for elem in self.items:
            if elem == base:
                continue
            comuni = base_ingredienti.intersection(set(i.lower() for i in elem["ingredienti"]))
            if len(comuni) >= soglia:
                suggerimenti.append({"nome": elem["nome"], "comuni": list(comuni)})
        logging.info(f"Trovati {len(suggerimenti)} suggerimenti per '{base['nome']}'")
        return suggerimenti

# INTERFACCIA GRAFICA
class ReFreshFoodApp:
    def __init__(self, root: tk.Tk):
        self.manager = ReFreshFoodManager()
        self.root = root
        self.root.title("ReFresh Food Manager")
        self.root.geometry("950x600")

        # Frame superiore per input
        input_frame = ttk.LabelFrame(root, text="Gestione Ricette", padding=10)
        input_frame.pack(fill="x", padx=10, pady=5)

        ttk.Label(input_frame, text="Nome:").grid(row=0, column=0)
        self.entry_nome = ttk.Entry(input_frame, width=20)
        self.entry_nome.grid(row=0, column=1)

        ttk.Label(input_frame, text="Categoria:").grid(row=0, column=2)
        self.entry_categoria = ttk.Entry(input_frame, width=20)
        self.entry_categoria.grid(row=0, column=3)

        ttk.Label(input_frame, text="Tempo (min):").grid(row=1, column=0)
        self.entry_tempo = ttk.Entry(input_frame, width=10)
        self.entry_tempo.grid(row=1, column=1)

        ttk.Label(input_frame, text="Ingredienti (separati da virgola):").grid(row=1, column=2)
        self.entry_ingredienti = ttk.Entry(input_frame, width=40)
        self.entry_ingredienti.grid(row=1, column=3)

        ttk.Button(input_frame, text="Aggiungi", command=self.aggiungi).grid(row=2, column=0, pady=5)
        ttk.Button(input_frame, text="Elimina selezionato", command=self.elimina).grid(row=2, column=1)
        ttk.Button(input_frame, text="Statistiche", command=self.mostra_statistiche).grid(row=2, column=2)
        ttk.Button(input_frame, text="Filtro Avanzato", command=self.filtro_popup).grid(row=2, column=3)
        ttk.Button(input_frame, text="Suggerisci simili", command=self.suggerisci_simili).grid(row=2, column=4)

        # Campo ricerca
        search_frame = ttk.Frame(root)
        search_frame.pack(fill="x", padx=10, pady=5)
        ttk.Label(search_frame, text="Cerca:").pack(side="left")
        self.entry_ricerca = ttk.Entry(search_frame, width=30)
        self.entry_ricerca.pack(side="left", padx=5)
        ttk.Button(search_frame, text="Cerca", command=self.cerca).pack(side="left", padx=5)
        ttk.Button(search_frame, text="Mostra Tutto", command=self.mostra_tutti).pack(side="left")

        # Ordinamento
        ttk.Label(search_frame, text="Ordina per:").pack(side="left", padx=(30, 5))
        self.sort_option = ttk.Combobox(search_frame, values=["nome", "categoria", "tempo_preparazione"], width=18)
        self.sort_option.current(0)
        self.sort_option.pack(side="left")
        ttk.Button(search_frame, text="Ordina", command=self.ordina).pack(side="left", padx=5)

        # Tabella
        self.tree = ttk.Treeview(root, columns=("Nome", "Categoria", "Tempo", "Ingredienti"), show="headings")
        self.tree.pack(fill="both", expand=True, padx=10, pady=10)
        for col in ("Nome", "Categoria", "Tempo", "Ingredienti"):
            self.tree.heading(col, text=col)
            self.tree.column(col, width=180)
        self.mostra_tutti()

    # Metodi GUI
    def aggiorna_tabella(self, dati: List[Dict[str, any]]) -> None:
        for i in self.tree.get_children():
            self.tree.delete(i)
        for elem in dati:
            self.tree.insert("", "end", values=(
                elem["nome"],
                elem["categoria"],
                elem["tempo_preparazione"],
                ", ".join(elem["ingredienti"])
            ))

    def mostra_tutti(self) -> None:
        self.aggiorna_tabella(self.manager.items)

    def aggiungi(self) -> None:
        try:
            nome = self.entry_nome.get()
            categoria = self.entry_categoria.get()
            tempo = int(self.entry_tempo.get())
            ingredienti = [i.strip() for i in self.entry_ingredienti.get().split(",")]
            self.manager.aggiungi_elemento(nome, ingredienti, categoria, tempo)
            self.mostra_tutti()
            messagebox.showinfo("Successo", f"Ricetta '{nome}' aggiunta con successo.")
        except ValueError:
            messagebox.showerror("Errore", "Tempo deve essere un numero intero.")

    def elimina(self) -> None:
        sel = self.tree.selection()
        if sel:
            index = self.tree.index(sel[0])
            self.manager.elimina_elemento(index)
            self.mostra_tutti()

    def cerca(self) -> None:
        testo = self.entry_ricerca.get().strip()
        if testo:
            risultati = self.manager.ricerca(testo)
            self.aggiorna_tabella(risultati)

    def ordina(self) -> None:
        chiave = self.sort_option.get()
        self.manager.ordina(chiave)
        self.mostra_tutti()

    def mostra_statistiche(self) -> None:
        stats = self.manager.statistiche()
        if not stats:
            messagebox.showinfo("Statistiche", "Nessuna ricetta presente.")
            return
        msg = (
            f"Totale ricette: {stats['totale']}\n"
            f"Tempo medio: {stats['media_tempo']:.1f} min\n"
            f"Tempo minimo: {stats['min_tempo']} min\n"
            f"Tempo massimo: {stats['max_tempo']} min\n\n"
            "Distribuzione per categoria:\n"
        )
        for cat, count in stats['categorie'].items():
            msg += f"- {cat}: {count}\n"
        messagebox.showinfo("Statistiche", msg)

    def filtro_popup(self) -> None:
        popup = tk.Toplevel(self.root)
        popup.title("Filtro Avanzato")
        popup.geometry("300x200")

        ttk.Label(popup, text="Tempo massimo (min):").pack()
        entry_tempo = ttk.Entry(popup)
        entry_tempo.pack()

        ttk.Label(popup, text="Categoria:").pack()
        entry_categoria = ttk.Entry(popup)
        entry_categoria.pack()

        ttk.Label(popup, text="Ingrediente:").pack()
        entry_ingrediente = ttk.Entry(popup)
        entry_ingrediente.pack()

        def applica():
            max_tempo = int(entry_tempo.get()) if entry_tempo.get() else None
            categoria = entry_categoria.get() or None
            ingrediente = entry_ingrediente.get() or None
            risultati = self.manager.filtro_avanzato(max_tempo, categoria, ingrediente)
            self.aggiorna_tabella(risultati)
            popup.destroy()

        ttk.Button(popup, text="Applica", command=applica).pack(pady=10)

    def suggerisci_simili(self) -> None:
        sel = self.tree.selection()
        if not sel:
            messagebox.showinfo("Suggerimenti", "Seleziona una ricetta per vedere suggerimenti simili.")
            return

        index = self.tree.index(sel[0])
        suggerimenti = self.manager.suggerisci_simili(index)

        if not suggerimenti:
            messagebox.showinfo("Suggerimenti", "Nessuna ricetta simile trovata.")
            return

        msg = "Ricette simili trovate:\n\n"
        for s in suggerimenti:
            comuni = ", ".join(s["comuni"])
            msg += f"- {s['nome']} (ingredienti in comune: {comuni})\n"

        messagebox.showinfo("Suggerimenti", msg)

# AVVIO APP
if __name__ == "__main__":
    root = tk.Tk()
    app = ReFreshFoodApp(root)
    root.mainloop()